In [14]:
import datetime as dt
import dask.dataframe as dd
import pandas as pd
import dask
from sqlalchemy import select, create_engine
from sqlalchemy.engine import Engine
from sqlalchemy.sql.expression import func
from sqlalchemy.sql.expression import literal_column, literal
from sqlalchemy.dialects.postgresql import INTERVAL
from dotenv import load_dotenv
import os
from statsmodels.tsa.stattools import coint
import mc_postgres_db.models as models
from sqlalchemy.orm import Session
from dask.distributed import Client

load_dotenv()

POSTGRES_URL = os.getenv("POSTGRES_URL")

engine = create_engine(POSTGRES_URL)

In [2]:
client = Client(n_workers=8)
display(client)

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 8
Total threads: 16,Total memory: 32.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:51947,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:51966,Total threads: 2
Dashboard: http://127.0.0.1:51968/status,Memory: 4.00 GiB
Nanny: tcp://127.0.0.1:51950,


2025-10-30 20:43:55,837 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 20c9d4e796e623205bae40411084f833 initialized by task ('shuffle-transfer-20c9d4e796e623205bae40411084f833', 0) executed on worker tcp://127.0.0.1:51967
2025-10-30 20:43:58,853 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 20c9d4e796e623205bae40411084f833 deactivated due to stimulus 'task-finished-1761857038.851609'


In [21]:
date: dt.date = dt.datetime.now(dt.timezone.utc).date() - dt.timedelta(days=1)
end = dt.datetime.combine(date, dt.time.min)
start = end - dt.timedelta(days=7)
start_naive = start.replace(tzinfo=None).replace(second=0, microsecond=0)
end_naive = end.replace(tzinfo=None).replace(second=0, microsecond=0)
print(f"Start: {start}, End: {end}")

Start: 2025-10-22 00:00:00, End: 2025-10-29 00:00:00


In [23]:
max_groups = 10
with Session(engine) as session:
    # Get all provider asset group id(s)
    provider_asset_group_ids = session.scalars(
        select(models.ProviderAssetGroup.id).limit(max_groups)
    ).all()
print(
    f"Provider asset group ids (count: {len(provider_asset_group_ids)}): {provider_asset_group_ids}"
)

Provider asset group ids (count: 10): [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [22]:
# Create subquery that generates minutely timestamps using PostgreSQL's generate_series
start_str = start_naive.strftime("'%Y-%m-%d %H:%M:%S'::timestamp")
end_str = end_naive.strftime("'%Y-%m-%d %H:%M:%S'::timestamp")
time_frame_subquery = select(
    func.generate_series(
        literal_column(start_str),
        literal_column(end_str),
        func.cast(literal("1 minute"), INTERVAL),
    ).label("timestamp")
).subquery("time_frame")

# Create the sub-query for all provider asset group members.
provider_asset_group_members_subquery = (
    select(
        models.ProviderAssetGroupMember.provider_asset_group_id,
        models.ProviderAssetGroupMember.order,
        models.ProviderAssetGroupMember.provider_id,
        models.ProviderAssetGroupMember.from_asset_id,
        models.ProviderAssetGroupMember.to_asset_id,
    )
    .where(
        models.ProviderAssetGroupMember.provider_asset_group_id.in_(
            provider_asset_group_ids
        )
    )
    .subquery("provider_asset_group_members")
)

# Combine the time-frame with the provider asset group members into a dask dataframe.
# Using cross join to create a complete timeframe for each asset group member
full_frame: dd.DataFrame = dd.read_sql_query(
    select(
        time_frame_subquery.c.timestamp,
        provider_asset_group_members_subquery.c.provider_asset_group_id,
        provider_asset_group_members_subquery.c.order,
        provider_asset_group_members_subquery.c.provider_id,
        provider_asset_group_members_subquery.c.from_asset_id,
        provider_asset_group_members_subquery.c.to_asset_id,
    )
    .select_from(
        time_frame_subquery.join(
            provider_asset_group_members_subquery, literal(True), isouter=False
        )
    )
    .order_by(time_frame_subquery.c.timestamp),
    engine.url.render_as_string(hide_password=False),
    index_col="timestamp",
    bytes_per_chunk="512 MiB",
)

# Get the market data Dask dataframe.
market_data: dd.DataFrame = dd.read_sql_query(
    select(
        models.ProviderAssetMarket.timestamp,
        models.ProviderAssetMarket.provider_id,
        models.ProviderAssetMarket.from_asset_id,
        models.ProviderAssetMarket.to_asset_id,
        models.ProviderAssetMarket.close,
    )
    .where(models.ProviderAssetMarket.timestamp.between(start_naive, end_naive))
    .order_by(models.ProviderAssetMarket.timestamp),
    engine.url.render_as_string(hide_password=False),
    index_col="timestamp",
    bytes_per_chunk="512 MiB",
)

# As-of join the market data with the full frame.
full_frame = dd.merge_asof(
    full_frame,
    market_data,
    left_index=True,
    right_index=True,
    by=["provider_id", "from_asset_id", "to_asset_id"],
)

# Split out the close for each order and create pairs-trading frame.
close_1 = full_frame.loc[
    full_frame["order"] == 1,
    ["provider_asset_group_id", "provider_id", "from_asset_id", "to_asset_id", "close"],
]
close_2 = full_frame.loc[
    full_frame["order"] == 2,
    ["provider_asset_group_id", "provider_id", "from_asset_id", "to_asset_id", "close"],
]
pairs_trading_frame = dd.merge(
    close_1,
    close_2,
    on=["timestamp", "provider_asset_group_id"],
    how="inner",
    suffixes=("_1", "_2"),
)